In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import networkx as nx

PROJECT_DIR = "/content/drive/MyDrive/Cancer Evolution Arena"
DATA_DIR = f"{PROJECT_DIR}/Data"
RESULTS_DIR = f"{DATA_DIR}/Results"

progression_path = f"{RESULTS_DIR}/notebook07_inferred_progression_edges.csv"

print("Progression file exists:", os.path.exists(progression_path))

Mounted at /content/drive
Progression file exists: True


In [2]:
progression_edges = pd.read_csv(progression_path)

print(progression_edges.shape)
progression_edges.head()

(9, 10)


,From,To,P_From,P_To,P_To_given_From,P_To_given_Not_From,Probability_Gain,CoOccurrence,OddsRatio,PValue
0,BRAF,PIK3CA,0.095652,0.073913,0.272727,0.052885,0.219843,6,6.715909,0.002373
1,ERBB4,SMARCA4,0.100000,0.056522,0.217391,0.038647,0.178744,5,6.909722,0.004841
2,KRAS,STK11,0.326087,0.173913,0.280000,0.122581,0.157419,21,2.783626,0.004966
3,STK11,ATM,0.173913,0.100000,0.225000,0.073684,0.151316,9,3.649770,0.007722
4,TP53,NF1,0.465217,0.130435,0.196262,0.073171,0.123091,21,3.093023,0.006245


In [3]:
G_prog = nx.DiGraph()

for _, row in progression_edges.iterrows():
    G_prog.add_edge(
        row["From"],
        row["To"],
        weight=row["Probability_Gain"],
        odds_ratio=row["OddsRatio"],
        p_value=row["PValue"],
        cooccurrence=row["CoOccurrence"]
    )

print("Nodes:", G_prog.number_of_nodes())
print("Edges:", G_prog.number_of_edges())

Nodes: 13
Edges: 9


In [4]:
def predict_next_mutations(current_state, top_n=3):
    """
    Predict likely next mutations from a current tumor mutation state.
    current_state: list of mutation genes, e.g. ["KRAS"]
    """
    current_state = set(current_state)
    candidates = {}

    for gene in current_state:
        if gene not in G_prog:
            continue

        for next_gene in G_prog.successors(gene):
            if next_gene in current_state:
                continue

            edge = G_prog[gene][next_gene]
            score = edge["weight"]

            if next_gene not in candidates:
                candidates[next_gene] = {
                    "score": 0,
                    "sources": [],
                    "odds_ratios": [],
                    "p_values": [],
                    "cooccurrences": []
                }

            candidates[next_gene]["score"] += score
            candidates[next_gene]["sources"].append(gene)
            candidates[next_gene]["odds_ratios"].append(edge["odds_ratio"])
            candidates[next_gene]["p_values"].append(edge["p_value"])
            candidates[next_gene]["cooccurrences"].append(edge["cooccurrence"])

    ranked = sorted(
        candidates.items(),
        key=lambda x: x[1]["score"],
        reverse=True
    )

    return ranked[:top_n]

In [5]:
predict_next_mutations(["KRAS"])

[('STK11',
  {'score': 0.1574193548387097,
   'sources': ['KRAS'],
   'odds_ratios': [2.783625730994152],
   'p_values': [0.0049660183597378],
   'cooccurrences': [21]}),
 ('RBM10',
  {'score': 0.1083870967741935,
   'sources': ['KRAS'],
   'odds_ratios': [3.5],
   'p_values': [0.0108285253526938],
   'cooccurrences': [12]})]

In [6]:
predict_next_mutations(["TP53"])

[('NF1',
  {'score': 0.1230909505356735,
   'sources': ['TP53'],
   'odds_ratios': [3.0930232558139537],
   'p_values': [0.0062446329891356],
   'cooccurrences': [21]}),
 ('ALK',
  {'score': 0.1100980168680191,
   'sources': ['TP53'],
   'odds_ratios': [3.683333333333333],
   'p_values': [0.0073777074282337],
   'cooccurrences': [17]}),
 ('EGFR',
  {'score': 0.0999164197249449,
   'sources': ['TP53'],
   'odds_ratios': [2.1900452488687785],
   'p_values': [0.043022866459094],
   'cooccurrences': [22]})]

In [7]:
predict_next_mutations(["BRAF"])

[('PIK3CA',
  {'score': 0.2198426573426573,
   'sources': ['BRAF'],
   'odds_ratios': [6.715909090909091],
   'p_values': [0.002372697642067],
   'cooccurrences': [6]})]

In [8]:
def print_forecast(current_state, top_n=3):
    predictions = predict_next_mutations(current_state, top_n=top_n)

    print("[SIMULATION RUN]")
    print("Initial State:", " + ".join(current_state))

    if len(predictions) == 0:
        print("No predicted next mutations from current progression graph.")
        return

    print("\nPredicted Next Evolutionary Steps:")

    total_score = sum(info["score"] for _, info in predictions)

    for i, (gene, info) in enumerate(predictions, start=1):
        probability = info["score"] / total_score if total_score > 0 else 0

        print(
            f"{i}. Add {gene} "
            f"(Relative Probability: {probability:.2%})"
        )
        print(f"   Supported by: {', '.join(info['sources'])}")
        print(f"   Combined Score: {info['score']:.3f}")
        print(f"   Mean Odds Ratio: {np.mean(info['odds_ratios']):.2f}")
        print(f"   Mean p-value: {np.mean(info['p_values']):.4f}")

In [9]:
print_forecast(["KRAS"])

[SIMULATION RUN]
Initial State: KRAS

Predicted Next Evolutionary Steps:
1. Add STK11 (Relative Probability: 59.22%)
   Supported by: KRAS
   Combined Score: 0.157
   Mean Odds Ratio: 2.78
   Mean p-value: 0.0050
2. Add RBM10 (Relative Probability: 40.78%)
   Supported by: KRAS
   Combined Score: 0.108
   Mean Odds Ratio: 3.50
   Mean p-value: 0.0108


In [10]:
def simulate_evolution(start_state, steps=2, top_n=2):
    """
    Builds a branching simulation tree for possible tumor evolution.
    """
    branches = [{
        "state": list(start_state),
        "path": [list(start_state)],
        "score": 1.0
    }]

    for step in range(steps):
        new_branches = []

        for branch in branches:
            current_state = branch["state"]
            predictions = predict_next_mutations(current_state, top_n=top_n)

            if len(predictions) == 0:
                new_branches.append(branch)
                continue

            total_score = sum(info["score"] for _, info in predictions)

            for gene, info in predictions:
                relative_prob = info["score"] / total_score if total_score > 0 else 0

                new_state = sorted(list(set(current_state + [gene])))

                new_branches.append({
                    "state": new_state,
                    "path": branch["path"] + [new_state],
                    "score": branch["score"] * relative_prob
                })

        branches = new_branches

    branches = sorted(
        branches,
        key=lambda x: x["score"],
        reverse=True
    )

    return branches

In [11]:
def print_simulation_tree(start_state, steps=2, top_n=2):
    branches = simulate_evolution(
        start_state=start_state,
        steps=steps,
        top_n=top_n
    )

    print("[TUMOR EVOLUTION SIMULATION]")
    print("Starting State:", " + ".join(start_state))
    print()

    for i, branch in enumerate(branches, start=1):
        print(f"Branch {i} | Relative Path Probability: {branch['score']:.2%}")

        for step_num, state in enumerate(branch["path"]):
            label = "Initial" if step_num == 0 else f"Step {step_num}"
            print(f"  {label}: {' + '.join(state)}")

        print()

In [12]:
print_simulation_tree(["KRAS"], steps=2, top_n=2)

[TUMOR EVOLUTION SIMULATION]
Starting State: KRAS

Branch 1 | Relative Path Probability: 40.78%
  Initial: KRAS
  Step 1: KRAS + RBM10
  Step 2: KRAS + RBM10 + STK11

Branch 2 | Relative Path Probability: 34.51%
  Initial: KRAS
  Step 1: KRAS + STK11
  Step 2: ATM + KRAS + STK11

Branch 3 | Relative Path Probability: 24.72%
  Initial: KRAS
  Step 1: KRAS + STK11
  Step 2: KRAS + RBM10 + STK11



In [13]:
print_simulation_tree(["TP53"], steps=2, top_n=2)

[TUMOR EVOLUTION SIMULATION]
Starting State: TP53

Branch 1 | Relative Path Probability: 27.67%
  Initial: TP53
  Step 1: NF1 + TP53
  Step 2: ALK + NF1 + TP53

Branch 2 | Relative Path Probability: 26.06%
  Initial: TP53
  Step 1: ALK + TP53
  Step 2: ALK + NF1 + TP53

Branch 3 | Relative Path Probability: 25.11%
  Initial: TP53
  Step 1: NF1 + TP53
  Step 2: EGFR + NF1 + TP53

Branch 4 | Relative Path Probability: 21.15%
  Initial: TP53
  Step 1: ALK + TP53
  Step 2: ALK + EGFR + TP53



In [14]:
print_simulation_tree(["BRAF"], steps=2, top_n=2)

[TUMOR EVOLUTION SIMULATION]
Starting State: BRAF

Branch 1 | Relative Path Probability: 100.00%
  Initial: BRAF
  Step 1: BRAF + PIK3CA



In [15]:
branches_kras = simulate_evolution(["KRAS"], steps=2, top_n=2)
branches_tp53 = simulate_evolution(["TP53"], steps=2, top_n=2)
branches_braf = simulate_evolution(["BRAF"], steps=2, top_n=2)

sim_rows = []

for start_gene, branches in {
    "KRAS": branches_kras,
    "TP53": branches_tp53,
    "BRAF": branches_braf
}.items():
    for i, branch in enumerate(branches, start=1):
        sim_rows.append({
            "Start": start_gene,
            "Branch": i,
            "Final_State": " + ".join(branch["state"]),
            "Relative_Path_Probability": branch["score"],
            "Path": " -> ".join([" + ".join(state) for state in branch["path"]])
        })

simulation_results = pd.DataFrame(sim_rows)

simulation_results.to_csv(
    f"{RESULTS_DIR}/notebook08_simulation_results.csv",
    index=False
)

simulation_results

,Start,Branch,Final_State,Relative_Path_Probability,Path
0,KRAS,1,KRAS + RBM10 + STK11,0.407767,KRAS -> KRAS + RBM10 -> KRAS + RBM10 + STK11
1,KRAS,2,ATM + KRAS + STK11,0.345064,KRAS -> KRAS + STK11 -> ATM + KRAS + STK11
2,KRAS,3,KRAS + RBM10 + STK11,0.247169,KRAS -> KRAS + STK11 -> KRAS + RBM10 + STK11
3,TP53,1,ALK + NF1 + TP53,0.276725,TP53 -> NF1 + TP53 -> ALK + NF1 + TP53
4,TP53,2,ALK + NF1 + TP53,0.260602,TP53 -> ALK + TP53 -> ALK + NF1 + TP53
5,TP53,3,EGFR + NF1 + TP53,0.251134,TP53 -> NF1 + TP53 -> EGFR + NF1 + TP53
6,TP53,4,ALK + EGFR + TP53,0.211538,TP53 -> ALK + TP53 -> ALK + EGFR + TP53
7,BRAF,1,BRAF + PIK3CA,1.000000,BRAF -> BRAF + PIK3CA
